In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1. Inicialização do PySpark
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

conf = SparkConf()
conf.set('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.11.901')
conf.set('spark.hadoop.fs.s3a.aws.credentials.provider', 'com.amazonaws.auth.InstanceProfileCredentialsProvider')

spark = SparkSession.builder.config(conf=conf).getOrCreate()

:: loading settings :: url = jar:file:/usr/local/lib/python3.7/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-cef1ab38-9354-4969-b0fe-7b39af75dd34;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 475ms :: artifacts dl 22ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 by [com.amazonaws#aws-java-sdk-bundle;1.12.262] in [default]
	---------------------------------------------------------------------
	|     

In [3]:

from pyspark.sql import functions as F

# Caminhos dos arquivos no S3
customers_path = "s3a://last-mile-optimization-trusted-sofhia/dataset-orders/customers_cleaned_dataset.csv/"
geolocation_path = "s3a://last-mile-optimization-trusted-sofhia/dataset-orders/geolocation_cleaned_dataset.csv/"

# Leitura dos datasets
customers_df = spark.read.csv(
    customers_path,
    header=True,
    inferSchema=True
)

geolocation_df = spark.read.csv(
    geolocation_path,
    header=True,
    inferSchema=True
)

# 1. Calcular uma coordenada representativa por CEP
geo_por_cep = (
    geolocation_df
    .filter(
        F.col("geolocation_zip_code_prefix").isNotNull() &
        F.col("geolocation_lat").isNotNull() &
        F.col("geolocation_lng").isNotNull()
    )
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        F.percentile_approx(
            "geolocation_lat", 0.5
        ).alias("geolocation_lat"),

        F.percentile_approx(
            "geolocation_lng", 0.5
        ).alias("geolocation_lng")
    )
    .withColumnRenamed(
        "geolocation_zip_code_prefix",
        "customer_zip_code_prefix"
    )
)

# 2. Associar cada cliente à coordenada de seu CEP
joined_df = customers_df.join(
    geo_por_cep,
    on="customer_zip_code_prefix",
    how="inner"
)

# 3. Auditoria
print("=== AUDITORIA CUSTOMERS + GEOLOCATION ===")

print("Clientes antes do join:", customers_df.count())
print("Clientes após o join:", joined_df.count())

print(
    "Clientes sem coordenadas:",
    customers_df.join(
        geo_por_cep,
        on="customer_zip_code_prefix",
        how="left_anti"
    ).count()
)

print(
    "CEPs duplicados na tabela agregada:",
    geo_por_cep.groupBy("customer_zip_code_prefix")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Clientes duplicados após join:",
    joined_df.groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

joined_df.show(5)
joined_df.printSchema()


26/11/03 01:06:25 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


=== AUDITORIA CUSTOMERS + GEOLOCATION ===
Clientes antes do join: 41746


Clientes após o join: 41731


Clientes sem coordenadas: 15


CEPs duplicados na tabela agregada: 0


Clientes duplicados após join: 0


+------------------------+--------------------+--------------------+-------------+------------------+------------------+
|customer_zip_code_prefix|         customer_id|  customer_unique_id|customer_city|   geolocation_lat|   geolocation_lng|
+------------------------+--------------------+--------------------+-------------+------------------+------------------+
|                    1005|6e4eb34e7f4d526a8...|ded4351942c7fc292...|    SAO PAULO|-23.54961248124544|-46.63653202553522|
|                    1005|15d7dbcd027b5b248...|ddab5650ba76e2fa2...|    SAO PAULO|-23.54961248124544|-46.63653202553522|
|                    1005|c46061cd5153f3560...|84a7776f914ff1950...|    SAO PAULO|-23.54961248124544|-46.63653202553522|
|                    1005|f3e31afdae80581be...|ded4351942c7fc292...|    SAO PAULO|-23.54961248124544|-46.63653202553522|
|                    1005|6ec2b4682814cfdac...|57f0ea1c7f6b9ef86...|    SAO PAULO|-23.54961248124544|-46.63653202553522|
+------------------------+------

In [4]:
joined_df.coalesce(1) \
    .write \
    .option('header', 'true') \
    .mode('overwrite') \
    .csv('s3a://last-mile-optimization-trusted-sofhia/dataset-orders/join_customers_and_geolocation.csv')

spark.stop()

26/11/03 01:07:06 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/11/03 01:07:07 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
